In [35]:
from collections import Counter
import json, os, itertools

data_folder = "data/art_or_artifice/"

with open(f'{data_folder}humanannotationfinal.json') as f:
	data = json.load(f)


storymap = {'The Kingdom': 'The Kingdom That Failed','Returns':'Returns','MaintainenceH': 'Maintenance, Hvidovre','The Last Dance': 'The Last Dance with my Dad','Trash':'Trash','ListeningFC':'Listening For the Click','A Triangle': 'A Triangle', 'Keys': 'Keys','Barbara': 'Barbara,Detroit,1996','CertainEM': 'Certain European Movies','Beyond-Nat':'Beyond Nature','The Facade': "The Facade Renovation That_s Going Well" }

overall_ranks = {story_id: [] for story_id in storymap.values()}

for story in data:
    story_id = story['What is the story identifier (in the Google Doc)']
    story_id = story_id.replace("That's", "That_s").replace("Façade", "Facade")
    if "Facade" in story_id:
        story_id = "The Facade Renovation That_s Going Well"
    if 'Beyond-Nat' in story_id:
        story_id = 'Beyond Nature'
    elif '—' in story_id:
        story_id = storymap[story_id.split('—')[0]]
    elif '-' in story_id:
        story_id = storymap[story_id.split('-')[0]]

    with open(f'{data_folder}/teststories/{story_id}/sequencemap.json') as mp:
        order = {int(k): v for k,v in json.load(mp).items()}

    rank_texts = ["", "Most Preferred", "Second Most Preferred", "Third Most Preferred", "Least Favorite"]

    cleaned_ranks = {}
    for story_idx in range(1,5):
        model = order[story_idx]
        rank = rank_texts.index(story[f'Rank each of the four stories based on your preference [Story {story_idx}]'])
        cleaned_ranks[model] = rank

    overall_ranks[story_id].append(cleaned_ranks)

models = ["Claude.txt", "GPT3.5.txt", "NewYorker.txt", "GPT4.txt"]
AA_pairs = []
for story_id in overall_ranks:
    full_texts = {}
    for model in models:
        with open(f"{data_folder}/teststories/{story_id}/{model}") as f:
            full_texts[model] = f.read()

    for m1, m2 in itertools.combinations(models, 2):
        wins = []
        for clean_ranks in overall_ranks[story_id]:
            if clean_ranks[m1] < clean_ranks[m2]:
                wins.append(m1)
            else:
                wins.append(m2)
        win_counts = Counter(wins)
        winner, win_counts = win_counts.most_common(1)[0]
        win_counts /= len(wins)
        # print(f"{story_id.ljust(40)} {m1.ljust(15)} vs {m2.ljust(15)} {winner} {win_counts}")

        preference = "1" if winner == m1 else "2"
        sample = {"original_id": f"{story_id}_{m1}_{m2}", "model1": m1, "model2": m2, "story1": full_texts[m1], "story2": full_texts[m2], "winner": winner, "win_counts": win_counts, "preference": preference}
        AA_pairs.append(sample)

print(len(AA_pairs))
with open("data/art_or_artifice/AA_pairs_data.json", "w") as f:
    json.dump(AA_pairs, f, indent=4)


72


# Let's make the final version of the dataset with AB/BA

In [39]:
import json

with open("data/art_or_artifice/AA_pairs_data.json", "r") as f:
    AA_pairs = json.load(f)

with open("prompts/pairwise_pref.txt", "r") as f:
    pairwise_prompt = f.read()

# Make AB/BA pairs
all_pairs_data = []
for sample in AA_pairs:
    sample1 = {"id": f"test_artartifice_{len(all_pairs_data)}", "original_id": f"{sample['original_id']}", "sample_type": "pairwise-art", "model1": sample["model1"], "model2": sample["model2"], "story1": sample["story1"], "story2": sample["story2"], "reference_preference": sample["preference"], "reference_win_counts": sample["win_counts"]}
    sample1["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", sample1["story1"]).replace("[[PARAGRAPH2]]", sample1["story2"])
    all_pairs_data.append(sample1)

    reference_preference2 = "2" if sample["preference"] == "1" else "1"
    sample2 = {"id": f"test_artartifice_{len(all_pairs_data)}", "original_id": f"{sample['original_id']}", "sample_type": "pairwise-art", "model1": sample["model2"], "model2": sample["model1"], "story1": sample["story2"], "story2": sample["story1"], "reference_preference": reference_preference2, "reference_win_counts": sample["win_counts"]}
    sample2["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", sample2["story1"]).replace("[[PARAGRAPH2]]", sample2["story2"])
    all_pairs_data.append(sample2)

with open("data/art_or_artifice/AA_pairs_data.json", "w") as f:
    json.dump(all_pairs_data, f, indent=4)


In [40]:
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

lamp_PRGSH_test += all_pairs_data

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(lamp_PRGSH_test, f, indent=4)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

Counter({'pairwise-gold': 1206, 'pairwise-silver': 1120, 'pairwise-lmarena': 576, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-art': 144, 'pairwise-P7': 138})
